# 1. Transformaciones 2D aplicadas a puntos y rectas

**Tema 2 · Homografías (repaso) · PDF, pp. 4–5**

Imagina que fotografías una hoja cuadriculada desde un lado. La foto cambia la posición de sus puntos y puede hacer que dos líneas paralelas parezcan juntarse, pero una línea recta de la hoja sigue viéndose recta. Una **homografía** es la regla matemática que describe esa transformación del plano.

En este notebook moveremos un control para aumentar la perspectiva y veremos cómo una misma matriz transforma toda la cuadrícula, un punto y una recta.

> Ejecuta las celdas en orden. La visualización usa `numpy`, `plotly` e `ipywidgets` (si faltan: `pip install numpy plotly ipywidgets`).

## 1. La regla para transformar un punto

Representamos un punto del plano como `p = [x, y, 1]`. La matriz `H` lo lleva a `p′ ~ H p`. El símbolo `~` significa que las coordenadas pueden quedar multiplicadas por un mismo factor: para recuperar las coordenadas normales dividimos las dos primeras por la tercera.

En el gráfico, el panel izquierdo es el plano antes de la transformación y el derecho es el plano después. El punto rojo y la recta roja sirven para seguir ejemplos concretos.

In [4]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display


def aplicar_homografia(H, puntos):
    """Aplica H a puntos cartesianos (x, y) y vuelve a coordenadas cartesianas."""
    puntos = np.asarray(puntos, dtype=float)
    puntos_h = np.column_stack([puntos, np.ones(len(puntos))])
    resultado_h = (H @ puntos_h.T).T
    return resultado_h[:, :2] / resultado_h[:, 2:3]


def cerrar_poligono(puntos):
    return np.vstack([puntos, puntos[0]])


# Una cuadrícula sencilla y dos elementos para seguir su transformación.
xs = np.linspace(0, 4, 5)
ys = np.linspace(0, 3, 4)
esquinas = np.array([[0, 0], [4, 0], [4, 3], [0, 3]])
punto = np.array([[3.2, 2.3]])
recta_inicio = np.array([0.4, 0.5])
recta_fin = np.array([3.7, 2.1])

perspectiva = widgets.FloatSlider(
    value=1.0, min=0.0, max=1.5, step=0.05,
    description='perspectiva λ', readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='430px'),
)
salida = widgets.Output()
resumen = widgets.HTML()


def actualizar_grafico(*_):
    lam = perspectiva.value
    # λ=0 deja la cuadrícula intacta. Al crecer λ, aparece la perspectiva.
    H = np.array([
        [1.0, 0.0, 0.0],
        [0.0, 1.0, 0.0],
        [0.13 * lam, 0.07 * lam, 1.0],
    ])
    esquinas_despues = aplicar_homografia(H, esquinas)
    punto_despues = aplicar_homografia(H, punto)[0]
    recta_despues = aplicar_homografia(H, np.array([recta_inicio, recta_fin]))

    # Una recta se representa por l=[a,b,c], con a*x+b*y+c=0.
    # Su transformación correcta es l′ ~ (H⁻¹)ᵀ l.
    recta_h = np.cross(
        np.r_[recta_inicio, 1.0],
        np.r_[recta_fin, 1.0],
    )
    recta_h_despues = np.linalg.inv(H).T @ recta_h
    recta_h_despues /= np.linalg.norm(recta_h_despues[:2])
    extremos_h = np.column_stack([recta_despues, np.ones(2)])
    error_recta = np.max(np.abs(extremos_h @ recta_h_despues))

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Antes: plano cuadriculado', 'Después: plano transformado'),
        horizontal_spacing=0.12,
    )

    # Familia de líneas verticales.
    for i, x in enumerate(xs):
        original = np.array([[x, 0], [x, 3]])
        transformado = aplicar_homografia(H, original)
        fig.add_trace(go.Scatter(
            x=original[:, 0], y=original[:, 1], mode='lines',
            line=dict(color='#3182bd', width=1.5),
            name='líneas verticales', legendgroup='verticales',
            showlegend=(i == 0), hoverinfo='skip',
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=transformado[:, 0], y=transformado[:, 1], mode='lines',
            line=dict(color='#3182bd', width=1.5),
            name='líneas verticales', legendgroup='verticales',
            showlegend=False, hoverinfo='skip',
        ), row=1, col=2)

    # Familia de líneas horizontales.
    for i, y in enumerate(ys):
        original = np.array([[0, y], [4, y]])
        transformado = aplicar_homografia(H, original)
        fig.add_trace(go.Scatter(
            x=original[:, 0], y=original[:, 1], mode='lines',
            line=dict(color='#f28e2b', width=1.5),
            name='líneas horizontales', legendgroup='horizontales',
            showlegend=(i == 0), hoverinfo='skip',
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=transformado[:, 0], y=transformado[:, 1], mode='lines',
            line=dict(color='#f28e2b', width=1.5),
            name='líneas horizontales', legendgroup='horizontales',
            showlegend=False, hoverinfo='skip',
        ), row=1, col=2)

    for col, borde, p, extremos in [
        (1, esquinas, punto[0], np.array([recta_inicio, recta_fin])),
        (2, esquinas_despues, punto_despues, recta_despues),
    ]:
        borde_cerrado = cerrar_poligono(borde)
        fig.add_trace(go.Scatter(
            x=borde_cerrado[:, 0], y=borde_cerrado[:, 1], mode='lines',
            line=dict(color='#334155', width=2), name='borde del plano',
            showlegend=(col == 1), hoverinfo='skip',
        ), row=1, col=col)
        fig.add_trace(go.Scatter(
            x=extremos[:, 0], y=extremos[:, 1], mode='lines+markers',
            line=dict(color='#d6275f', width=3), marker=dict(size=7),
            name='una recta y sus puntos', showlegend=(col == 1),
        ), row=1, col=col)
        fig.add_trace(go.Scatter(
            x=[p[0]], y=[p[1]], mode='markers+text',
            marker=dict(color='#d6275f', size=12),
            text=['P' if col == 1 else 'P′'], textposition='top center',
            name='punto P', showlegend=False,
        ), row=1, col=col)

    fig.update_xaxes(range=[-0.2, 4.3], title_text='x', scaleanchor='y', scaleratio=1, row=1, col=1)
    fig.update_yaxes(range=[-0.2, 3.3], title_text='y', row=1, col=1)
    fig.update_xaxes(range=[-0.2, 4.3], title_text='x′', scaleanchor='y2', scaleratio=1, row=1, col=2)
    fig.update_yaxes(range=[-0.2, 3.3], title_text='y′', row=1, col=2)
    fig.update_layout(
        width=920, height=500,
        title=dict(text='Una homografía mueve puntos y conserva las rectas', x=0.5),
        margin=dict(l=45, r=25, t=80, b=50),
        legend=dict(orientation='h', y=-0.16),
    )

    p_texto = f"({punto_despues[0]:.2f}, {punto_despues[1]:.2f})"
    resumen.value = (
        f"<b>Intensidad de perspectiva:</b> λ = {lam:.2f}<br>"
        f"<b>H</b> = [[1, 0, 0], [0, 1, 0], "
        f"[{0.13 * lam:.3f}, {0.07 * lam:.3f}, 1]]<br>"
        f"<b>P</b> = (3.20, 2.30) se convierte en <b>P′</b> = {p_texto}. "
        f"Los extremos de la recta transformada cumplen su ecuación "
        f"(error numérico ≈ {error_recta:.1e})."
    )
    with salida:
        salida.clear_output(wait=True)
        fig.show()


perspectiva.observe(actualizar_grafico, names='value')
display(widgets.VBox([perspectiva, resumen, salida]))
actualizar_grafico()

### Cómo leer la visualización

- **Azul y naranja:** dos familias de rectas de la cuadrícula. Después de la transformación siguen siendo rectas. Con perspectiva, las que antes eran paralelas pueden parecer dirigirse a un punto de fuga.
- **P y P′:** el mismo punto del plano antes y después de aplicar `H`.
- **Rojo:** una recta concreta. Sus dos extremos cambian de sitio, y la recta que los une continúa siendo recta.
- Con `λ = 0`, `H` es la identidad y no hay deformación. Al aumentar `λ`, aumenta el efecto de perspectiva.

## 2. ¿Por qué cambia así una recta?

Una recta se puede escribir como `a*x + b*y + c = 0`, o como `lᵀ p = 0` usando coordenadas homogéneas. Si los puntos cambian según `p′ ~ H p`, sus rectas cambian según

`l′ ~ (H⁻¹)ᵀ l`.

La razón intuitiva es que los puntos de la recta transformada deben seguir satisfaciendo la ecuación de esa recta. La matriz inversa transpuesta hace que esto se cumpla para todos los puntos de la línea a la vez.

### Qué conviene recordar

1. La homografía aplica **una misma regla matricial** a todos los puntos del plano.
2. **Rectas se transforman en rectas**, aunque cambien longitudes, ángulos o paralelismo.
3. Para el punto usamos `p′ ~ H p`; para la recta usamos `l′ ~ (H⁻¹)ᵀ l`.

Este es el repaso geométrico que necesitaremos antes de estudiar cómo se calcula `H` con DLT.

## 2. La matriz: nueve casillas, ocho parámetros independientes

La matriz `H` tiene nueve entradas. Pero en coordenadas homogéneas una matriz y cualquier múltiplo suyo no nulo representan la misma transformación: `H` y `k·H` producen el mismo punto después de dividir por la tercera coordenada.

Por eso podemos fijar una de las entradas como escala de referencia —a menudo `h₃₃ = 1`, si esa entrada no es cero— y quedan **ocho parámetros independientes**. El control de abajo cambia `k`: verás que cambian los números escritos en la matriz, mientras que la figura transformada no cambia.

Esta celda reutiliza las funciones y librerías de las celdas anteriores; ejecuta el notebook desde el principio.

In [5]:
# Un ejemplo de homografía con h₃₃ = 1 como escala de referencia.
H_base = np.array([
    [1.00,  0.12, 0.30],
    [-0.08, 0.95, 0.20],
    [0.10,  0.06, 1.00],
])

factor_escala = widgets.FloatSlider(
    value=1.0, min=0.25, max=3.0, step=0.05,
    description='factor k', readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px'),
)
salida_escala = widgets.Output()
matrices_html = widgets.HTML()

# Borde, dos líneas interiores y una diagonal para reconocer la forma.
figuras_plano = [
    np.array([[0, 0], [3, 0], [3, 2], [0, 2], [0, 0]]),
    np.array([[1.5, 0], [1.5, 2]]),
    np.array([[0, 1], [3, 1]]),
    np.array([[0, 0], [3, 2]]),
]


def formato_matriz(M):
    filas = ['[ ' + '   '.join(f'{v:7.3f}' for v in fila) + ' ]' for fila in M]
    return '<br>'.join(filas)


def actualizar_escala(*_):
    k = factor_escala.value
    H_k = k * H_base

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Superpuestas: H y k·H', 'La misma forma vista con k·H'),
        horizontal_spacing=0.12,
    )
    colores = ['#334155', '#3182bd', '#f28e2b', '#7c3aed']
    for i, curva in enumerate(figuras_plano):
        con_H = aplicar_homografia(H_base, curva)
        con_kH = aplicar_homografia(H_k, curva)
        fig.add_trace(go.Scatter(
            x=con_H[:, 0], y=con_H[:, 1], mode='lines+markers',
            line=dict(color=colores[i], width=4), marker=dict(size=5),
            name='H', legendgroup='H', showlegend=(i == 0),
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=con_kH[:, 0], y=con_kH[:, 1], mode='lines',
            line=dict(color='#d6275f', width=2, dash='dash'),
            name='k·H', legendgroup='kH', showlegend=(i == 0),
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=con_kH[:, 0], y=con_kH[:, 1], mode='lines+markers',
            line=dict(color=colores[i], width=4), marker=dict(size=5),
            name='k·H', showlegend=False,
        ), row=1, col=2)

    for col in [1, 2]:
        fig.update_xaxes(range=[-0.2, 3.5], title_text='x′', scaleanchor='y' if col == 1 else 'y2', scaleratio=1, row=1, col=col)
        fig.update_yaxes(range=[-0.2, 2.7], title_text='y′', row=1, col=col)
    fig.update_layout(
        width=920, height=470,
        title=dict(text='Cambiar la escala de H no cambia la transformación', x=0.5),
        margin=dict(l=45, r=25, t=80, b=45),
        legend=dict(orientation='h', y=-0.16),
    )

    diferencia = np.max(np.abs(
        aplicar_homografia(H_base, np.array([[0, 0], [3, 0], [3, 2], [0, 2]]))
        - aplicar_homografia(H_k, np.array([[0, 0], [3, 0], [3, 2], [0, 2]]))
    ))
    matrices_html.value = (
        f'<b>H (con h₃₃ = 1):</b><br><code>{formato_matriz(H_base)}</code><br><br>'
        f'<b>k·H (k = {k:.2f}):</b><br><code>{formato_matriz(H_k)}</code><br><br>'
        f'<b>Máxima diferencia entre las coordenadas transformadas:</b> {diferencia:.2e}'
    )
    with salida_escala:
        salida_escala.clear_output(wait=True)
        fig.show()


factor_escala.observe(actualizar_escala, names='value')
display(widgets.VBox([factor_escala, matrices_html, salida_escala]))
actualizar_escala()

## 3. Anatomía de H: qué hace cada bloque

Si fijamos la escala homogénea de abajo a la derecha en 1, podemos separar la matriz así:

`H = [[A, t], [vᵀ, 1]]`

Al aplicarla a un punto `p = [x, y]`, sus coordenadas pasan a ser `p′ = (A·p + t) / (vᵀ·p + 1)`. Esta forma permite distinguir tres efectos: `A` cambia la orientación y la forma; `t` desplaza el plano; `v` hace que la división dependa de la posición y produce perspectiva.

Elige un efecto en el menú y cambia su intensidad. La celda muestra el bloque de la matriz que actúa y la cuadrícula antes y después.

In [ ]:
modos = [
    'Rotación (A)',
    'Escala (A)',
    'Cizalladura (A)',
    'Traslación (t)',
    'Perspectiva (v)',
]
modo = widgets.Dropdown(
    options=modos, value=modos[0], description='bloque / efecto',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='340px'),
)
intensidad_bloque = widgets.FloatSlider(
    value=0.7, min=-1.0, max=1.0, step=0.05,
    description='intensidad', readout_format='.2f',
    continuous_update=False,
    layout=widgets.Layout(width='430px'),
)
salida_bloques = widgets.Output()
info_bloques = widgets.HTML()

xs_bloques = np.linspace(0, 4, 5)
ys_bloques = np.linspace(0, 3, 4)


def matriz_del_efecto(nombre, a):
    if nombre == 'Rotación (A)':
        angulo = 0.7 * a
        A = np.array([[np.cos(angulo), -np.sin(angulo)],
                      [np.sin(angulo),  np.cos(angulo)]])
        t = np.zeros(2)
        v = np.zeros(2)
        explicacion = 'A gira el plano alrededor del origen; no añade perspectiva.'
    elif nombre == 'Escala (A)':
        A = np.diag([1.0 + 0.45 * a, 1.0 - 0.30 * a])
        t = np.zeros(2)
        v = np.zeros(2)
        explicacion = 'A estira o encoge los ejes; el origen queda fijo.'
    elif nombre == 'Cizalladura (A)':
        A = np.array([[1.0, 0.55 * a], [0.0, 1.0]])
        t = np.zeros(2)
        v = np.zeros(2)
        explicacion = 'A inclina la cuadrícula; las rectas siguen siendo paralelas a sus familias.'
    elif nombre == 'Traslación (t)':
        A = np.eye(2)
        t = np.array([1.4 * a, 0.8 * a])
        v = np.zeros(2)
        explicacion = 't desplaza todos los puntos por el mismo vector.'
    else:
        A = np.eye(2)
        t = np.zeros(2)
        v = np.array([0.10 * a, 0.05 * a])
        explicacion = 'v hace que el divisor dependa de la posición: aparece perspectiva y puede perderse el paralelismo.'
    H = np.block([[A, t.reshape(2, 1)], [v.reshape(1, 2), np.ones((1, 1))]])
    return H, A, t, v, explicacion


def texto_bloque(nombre_bloque, bloque):
    if np.ndim(bloque) == 1:
        return '[' + ', '.join(f'{valor:.2f}' for valor in bloque) + ']'
    return '<br>'.join('[' + ', '.join(f'{valor:.2f}' for valor in fila) + ']' for fila in bloque)


def actualizar_bloques(*_):
    H, A, t, v, explicacion = matriz_del_efecto(modo.value, intensidad_bloque.value)
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Antes: plano original', 'Después: efecto seleccionado'),
        horizontal_spacing=0.12,
    )
    for x in xs_bloques:
        puntos = np.array([[x, 0.0], [x, 3.0]])
        despues = aplicar_homografia(H, puntos)
        fig.add_trace(go.Scatter(x=puntos[:, 0], y=puntos[:, 1], mode='lines',
                                 line=dict(color='#3182bd', width=1.5),
                                 name='familia vertical', showlegend=bool(x == xs_bloques[0]), hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter(x=despues[:, 0], y=despues[:, 1], mode='lines',
                                 line=dict(color='#3182bd', width=1.5),
                                 name='familia vertical', showlegend=False, hoverinfo='skip'), row=1, col=2)
    for y in ys_bloques:
        puntos = np.array([[0.0, y], [4.0, y]])
        despues = aplicar_homografia(H, puntos)
        fig.add_trace(go.Scatter(x=puntos[:, 0], y=puntos[:, 1], mode='lines',
                                 line=dict(color='#f28e2b', width=1.5),
                                 name='familia horizontal', showlegend=bool(y == ys_bloques[0]), hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter(x=despues[:, 0], y=despues[:, 1], mode='lines',
                                 line=dict(color='#f28e2b', width=1.5),
                                 name='familia horizontal', showlegend=False, hoverinfo='skip'), row=1, col=2)

    origen = np.array([[0.0, 0.0]])
    origen_despues = aplicar_homografia(H, origen)[0]
    fig.add_trace(go.Scatter(x=[0], y=[0], mode='markers+text',
                             marker=dict(color='#d6275f', size=10), text=['O'], textposition='bottom left',
                             name='origen', showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=[origen_despues[0]], y=[origen_despues[1]], mode='markers+text',
                             marker=dict(color='#d6275f', size=10), text=['O′'], textposition='bottom left',
                             name='origen transformado', showlegend=False), row=1, col=2)
    for col in [1, 2]:
        fig.update_xaxes(range=[-2.2, 6.0], title_text='x', scaleanchor='y' if col == 1 else 'y2', scaleratio=1, row=1, col=col)
        fig.update_yaxes(range=[-2.0, 5.0], title_text='y', row=1, col=col)
    fig.update_layout(
        width=980, height=530,
        title=dict(text='La matriz separa cambios lineales, desplazamiento y perspectiva', x=0.5),
        margin=dict(l=45, r=25, t=80, b=45),
        legend=dict(orientation='h', y=-0.16),
    )

    A_html = texto_bloque('A', A)
    t_html = texto_bloque('t', t)
    v_html = texto_bloque('v', v)
    info_bloques.value = (
        f'<b>H = [[A, t], [vᵀ, 1]]</b><br>'
        f'<b>A</b> (lineal) =<br><code>{A_html}</code><br>'
        f'<b>t</b> (traslación) = <code>{t_html}</code> &nbsp; '
        f'<b>vᵀ</b> (perspectiva) = <code>{v_html}</code><br>'
        f'<b>Lectura:</b> {explicacion}'
    )
    with salida_bloques:
        salida_bloques.clear_output(wait=True)
        fig.show()


modo.observe(actualizar_bloques, names='value')
intensidad_bloque.observe(actualizar_bloques, names='value')
display(widgets.VBox([modo, intensidad_bloque, info_bloques, salida_bloques]))
actualizar_bloques()